In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('BVBRC_genome_amr_simplified.csv')
df.shape

(83096, 4)

In [2]:
print(df['target'].value_counts())

target
0    57912
1    25184
Name: count, dtype: int64


In [3]:
df = df.drop(columns=['Genome Name'])

In [4]:
X = df[['Antibiotic', 'Strain Name']].copy()
y = df['target']

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate

In [13]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

strain_le = LabelEncoder()
# Fit on training data
strain_le.fit(X_train["Strain Name"])

# Transform training data
X_train_strain_le = strain_le.transform(X_train["Strain Name"])

# Transform test data, map unseen labels to -1
X_test_strain_le = X_test["Strain Name"].apply(
    lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else -1
).values

print(X_train_strain_le)
print(X_test_strain_le)

[1774  537 1427 ... 1288 1728 1319]
[ 968 1774  948 ...  171 1907  969]


In [16]:
from sklearn.preprocessing import OneHotEncoder

# Use sparse_output instead of sparse
ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

# Fit on training data
ohe.fit(X_train[["Antibiotic"]])

# Transform train and test
X_train_ohe = ohe.transform(X_train[["Antibiotic"]])
X_test_ohe = ohe.transform(X_test[["Antibiotic"]])

In [18]:
import pandas as pd

# Wrap LE arrays in DataFrame, keeping the original index
X_train_strain_df = pd.DataFrame(X_train_strain_le, columns=["Strain_LE"], index=X_train.index)
X_test_strain_df  = pd.DataFrame(X_test_strain_le, columns=["Strain_LE"], index=X_test.index)

# Make sure OHE outputs are also DataFrames (with column names)
X_train_ohe_df = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_train.index)
X_test_ohe_df  = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_test.index)

# Now concatenate
X_train_ml = pd.concat([X_train_strain_df, X_train_ohe_df], axis=1)
X_test_ml  = pd.concat([X_test_strain_df, X_test_ohe_df], axis=1)

print("ML input shape:", X_train_ml.shape)

ML input shape: (66476, 69)


In [30]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_ml, y_train)

print("Train accuracy:", rf.score(X_train_ml, y_train))
print("Test accuracy:", rf.score(X_test_ml, y_test))

Train accuracy: 0.840228052229376
Test accuracy: 0.7515643802647413


In [27]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_ml, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [31]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
def evaluate_model(y_true, y_pred, y_proba, model_name="Model"):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_proba)
    
    print(f"--- {model_name} Metrics ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"ROC-AUC:   {roc:.4f}")
    print("\n")
    
# Evaluate RF
evaluate_model(y_test, y_pred_rf, y_pred_rf_proba, "Random Forest")

# Evaluate XGB
evaluate_model(y_test, y_pred_xgb, y_pred_xgb_proba, "XGBoost")

--- Random Forest Metrics ---
Accuracy:  0.7511
F1 Score:  0.5583
Precision: 0.6041
Recall:    0.5190
ROC-AUC:   0.7903


--- XGBoost Metrics ---
Accuracy:  0.7486
F1 Score:  0.5314
Precision: 0.6107
Recall:    0.4703
ROC-AUC:   0.7968




In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Dense, Concatenate

# Features and target
X = df[["Strain Name","Antibiotic"]]
y = df["target"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

strain_le = LabelEncoder()
strain_le.fit(X_train["Strain Name"])

X_train_strain_idx = X_train["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values
X_test_strain_idx  = X_test["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values

# =========================
# Step 3: One-Hot Encode 'Antibiotic'
# =========================
ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")
ohe.fit(X_train[["Antibiotic"]])

X_train_ohe = ohe.transform(X_train[["Antibiotic"]])
X_test_ohe  = ohe.transform(X_test[["Antibiotic"]])

# =========================
# Step 4: Build Neural Network
# =========================
# Strain input embedding
strain_input = Input(shape=(1,), name="strain_input")
strain_embed = Embedding(input_dim=len(strain_le.classes_)+1, output_dim=50)(strain_input)
strain_flat = Flatten()(strain_embed)

# Antibiotic one-hot input
antibiotic_input = Input(shape=(X_train_ohe.shape[1],), name="antibiotic_input")

# Concatenate features
x = Concatenate()([strain_flat, antibiotic_input])
x = Dense(64, activation="relu")(x)
x = Dropout(0.3)
x = Dense(32, activation="relu")(x)
x = Dropout(0.2)
x = Dense(16, activation="tanh")(x)
output = Dense(1, activation="sigmoid")(x)

# Model
model = Model(inputs=[strain_input, antibiotic_input], outputs=output)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

# =========================
# Step 5: Train Model
# =========================
history = model.fit(
    [X_train_strain_idx, X_train_ohe],
    y_train.values,
    validation_split=0.2,
    epochs=20,
    batch_size=4,
    verbose=1
)

# =========================
# Step 6: Predict & Evaluate
# =========================
y_pred_proba = model.predict([X_test_strain_idx, X_test_ohe]).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

def evaluate_model_nn(y_true, y_pred, y_proba, model_name="NN Model"):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_proba)
    
    print(f"--- {model_name} Metrics ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"ROC-AUC:   {roc:.4f}")
    print("\n")

# Evaluate NN
evaluate_model_nn(y_test, y_pred, y_pred_proba, "Neural Network")

TypeError: OneHotEncoder.__init__() got an unexpected keyword argument 'sparse'

In [35]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Embedding, Flatten, Concatenate
from tensorflow.keras import regularizers, initializers
from keras_tuner import HyperModel

# Example HyperModel class
class MyHyperModel(HyperModel):
    def __init__(self, input_dim):
        self.input_dim = input_dim

    def build(self, hp):
        model = Sequential()
        counter = 0
        for i in range(hp.Int('num_layers', min_value=1, max_value=3)):
            units = hp.Int('units' + str(i), min_value=8, max_value=128, step=8)
            activation = hp.Choice('activation' + str(i), values=['relu','tanh','sigmoid'])
            dropout_rate = hp.Choice('dropout' + str(i), values=[0.1,0.2,0.3,0.4,0.5])
            reg = regularizers.l2(hp.Choice('l2_reg'+str(i), values=[0.0, 0.001,0.01,0.1]))
            kernel_init = hp.Choice('kernel_init'+str(i), values=['glorot_uniform','he_uniform','he_normal','lecun_normal'])
            
            if counter == 0:
                model.add(
                    Dense(
                        units,
                        activation=activation,
                        input_dim=self.input_dim,
                        kernel_regularizer=reg,
                        kernel_initializer=kernel_init
                    )
                )
                model.add(BatchNormalization())
                model.add(Dropout(dropout_rate))
            else:
                model.add(
                    Dense(
                        units,
                        activation=activation,
                        kernel_regularizer=reg,
                        kernel_initializer=kernel_init
                    )
                )
                model.add(BatchNormalization())
                model.add(Dropout(dropout_rate))
            counter += 1

        model.add(Dense(1, activation='sigmoid'))
        optimizer = hp.Choice('optimizer', values=['rmsprop','adam','sgd','nadam','adadelta','adamw'])
        model.compile(optimizer=optimizer,
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
        return model

In [38]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten

# Train-test split
X = df[["Strain Name","Antibiotic"]]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Label encode Strain_Name for embedding
strain_le = LabelEncoder()
strain_le.fit(X_train["Strain Name"])
X_train_strain_idx = X_train["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values
X_test_strain_idx  = X_test["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values

# Simple embedding model
input_strain = Input(shape=(1,))
embed_layer = Embedding(input_dim=len(strain_le.classes_)+1, output_dim=10, name="strain_embedding")(input_strain)
embed_flat = Flatten()(embed_layer)
embedding_model = Model(inputs=input_strain, outputs=embed_flat)

# Forward pass to get embeddings
train_embeddings = embedding_model.predict(X_train_strain_idx)
test_embeddings  = embedding_model.predict(X_test_strain_idx)

print("Train embeddings shape:", train_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

520/520 [==============================] - 1s 1ms/step
Train embeddings shape: (66476, 10)
Test embeddings shape: (16620, 10)


In [39]:
train_embeddings = embedding_model.predict(X_train_strain_idx)
test_embeddings  = embedding_model.predict(X_test_strain_idx)

520/520 [==============================] - 1s 1ms/step


In [41]:
from sklearn.preprocessing import OneHotEncoder

# Use sparse_output instead of sparse
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on training data
ohe.fit(X_train[["Antibiotic"]])

# Transform train and test
X_train_ohe = ohe.transform(X_train[["Antibiotic"]])
X_test_ohe  = ohe.transform(X_test[["Antibiotic"]])

# Optional: convert to DataFrame with column names
import pandas as pd
X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_train.index)
X_test_ohe  = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_test.index)

In [42]:
# Concatenate embeddings with OHE features
X_train_ml = np.hstack([train_embeddings, X_train_ohe])
X_test_ml  = np.hstack([test_embeddings, X_test_ohe])

input_dim = X_train_ml.shape[1]  # This is the input_dim for MyHyperModel

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Embedding, Flatten, Concatenate
from tensorflow.keras import regularizers, initializers
from kerastuner import HyperModel

# Example HyperModel class
class MyHyperModel(HyperModel):
    def __init__(self, input_dim):
        self.input_dim = input_dim

    def build(self, hp):
        model = Sequential()
        counter = 0
        for i in range(hp.Int('num_layers', min_value=1, max_value=5)):
            units = hp.Int('units' + str(i), min_value=8, max_value=128, step=8)
            activation = hp.Choice('activation' + str(i), values=['relu','tanh','sigmoid'])
            dropout_rate = hp.Choice('dropout' + str(i), values=[0.1,0.2,0.3,0.4,0.5])
            reg = regularizers.l2(hp.Choice('l2_reg'+str(i), values=[0.0, 0.001,0.01,0.1]))
            kernel_init = hp.Choice('kernel_init'+str(i), values=['glorot_uniform','he_uniform','he_normal','lecun_normal'])
            
            if counter == 0:
                model.add(
                    Dense(
                        units,
                        activation=activation,
                        input_dim=self.input_dim,
                        kernel_regularizer=reg,
                        kernel_initializer=kernel_init
                    )
                )
                model.add(BatchNormalization())
                model.add(Dropout(dropout_rate))
            else:
                model.add(
                    Dense(
                        units,
                        activation=activation,
                        kernel_regularizer=reg,
                        kernel_initializer=kernel_init
                    )
                )
                model.add(BatchNormalization())
                model.add(Dropout(dropout_rate))
            counter += 1

        model.add(Dense(1, activation='sigmoid'))
        optimizer = hp.Choice('optimizer', values=['rmsprop','adam','sgd','nadam','adadelta','adamw'])
        model.compile(optimizer=optimizer,
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
        return model

In [44]:
hypermodel = MyHyperModel(input_dim=input_dim)

In [46]:
tuner = kt.RandomSearch(
    hypermodel,                    
    objective='val_accuracy',      
    max_trials=5,                 
    directory='mydir',
    project_name='project10'
)

In [47]:
tuner.search(
    X_train_ml.astype(np.float32),  # feature matrix
    y_train.values.astype(np.float32),  # target vector
    epochs=5,
    validation_data=(X_test_ml.astype(np.float32), y_test.values.astype(np.float32)),
    verbose=1
)

Trial 5 Complete [00h 01m 25s]
val_accuracy: 0.7155836224555969

Best val_accuracy So Far: 0.7188928723335266
Total elapsed time: 00h 06m 27s


In [48]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:", best_hps.values)

Best Hyperparameters: {'num_layers': 2, 'units0': 8, 'activation0': 'tanh', 'dropout0': 0.2, 'l2_reg0': 0.01, 'kernel_init0': 'he_uniform', 'optimizer': 'nadam', 'units1': 8, 'activation1': 'relu', 'dropout1': 0.1, 'l2_reg1': 0.0, 'kernel_init1': 'glorot_uniform'}


In [49]:
best_model = tuner.get_best_models(num_models=1)[0]

In [50]:
history = best_model.fit(
    X_train_ml.astype(np.float32),
    y_train.values.astype(np.float32),
    epochs=15,
    initial_epoch=5,  # continue from end of hyperparameter search
    validation_data=(X_test_ml.astype(np.float32), y_test.values.astype(np.float32))
)

Epoch 6/15
2078/2078 [==============================] - 22s 10ms/step - loss: 0.5649 - accuracy: 0.7074 - val_loss: 0.5576 - val_accuracy: 0.7178
Epoch 7/15
2078/2078 [==============================] - 19s 9ms/step - loss: 0.5618 - accuracy: 0.7087 - val_loss: 0.5624 - val_accuracy: 0.7142
Epoch 8/15
2078/2078 [==============================] - 19s 9ms/step - loss: 0.5612 - accuracy: 0.7094 - val_loss: 0.5582 - val_accuracy: 0.7137
Epoch 9/15
2078/2078 [==============================] - 19s 9ms/step - loss: 0.5605 - accuracy: 0.7106 - val_loss: 0.5565 - val_accuracy: 0.7120
Epoch 10/15
2078/2078 [==============================] - 19s 9ms/step - loss: 0.5599 - accuracy: 0.7103 - val_loss: 0.5569 - val_accuracy: 0.7170
Epoch 11/15
2078/2078 [==============================] - 19s 9ms/step - loss: 0.5598 - accuracy: 0.7103 - val_loss: 0.5587 - val_accuracy: 0.7094
Epoch 12/15
2078/2078 [==============================] - 19s 9ms/step - loss: 0.5593 - accuracy: 0.7124 - val_loss: 0.5566 - va

In [51]:
y_pred_proba = best_model.predict(X_test_ml.astype(np.float32)).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

520/520 [==============================] - 1s 2ms/step
Accuracy: 0.7192539109506618
F1 Score: 0.3613468382151656
Precision: 0.5817540766857646
Recall: 0.26206075044669447
ROC-AUC: 0.7152306721163845


In [52]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten

# Train-test split
X = df[["Strain Name","Antibiotic"]]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Label encode Strain_Name for embedding
strain_le = LabelEncoder()
strain_le.fit(X_train["Strain Name"])
X_train_strain_idx = X_train["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values
X_test_strain_idx  = X_test["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values

# Simple embedding model
input_strain = Input(shape=(1,))
embed_layer = Embedding(input_dim=len(strain_le.classes_)+1, output_dim=50, name="strain_embedding")(input_strain)
embed_flat = Flatten()(embed_layer)
embedding_model = Model(inputs=input_strain, outputs=embed_flat)

# Forward pass to get embeddings
train_embeddings = embedding_model.predict(X_train_strain_idx)
test_embeddings  = embedding_model.predict(X_test_strain_idx)

print("Train embeddings shape:", train_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

520/520 [==============================] - 1s 1ms/step
Train embeddings shape: (66476, 50)
Test embeddings shape: (16620, 50)


In [53]:
from sklearn.preprocessing import OneHotEncoder

# Use sparse_output instead of sparse
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on training data
ohe.fit(X_train[["Antibiotic"]])

# Transform train and test
X_train_ohe = ohe.transform(X_train[["Antibiotic"]])
X_test_ohe  = ohe.transform(X_test[["Antibiotic"]])

# Optional: convert to DataFrame with column names
import pandas as pd
X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_train.index)
X_test_ohe  = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_test.index)

In [54]:
# Concatenate embeddings with OHE features
X_train_ml = np.hstack([train_embeddings, X_train_ohe])
X_test_ml  = np.hstack([test_embeddings, X_test_ohe])

input_dim = X_train_ml.shape[1]  # This is the input_dim for MyHyperModel

In [59]:
hypermodel = MyHyperModel(input_dim=input_dim)

In [60]:
tuner = kt.RandomSearch(
    hypermodel,                    
    objective='val_accuracy',      
    max_trials=10,                 
    directory='mydir',
    project_name='project15'
)

In [61]:
tuner.search(
    X_train_ml.astype(np.float32),  # feature matrix
    y_train.values.astype(np.float32),  # target vector
    epochs=5,
    validation_data=(X_test_ml.astype(np.float32), y_test.values.astype(np.float32)),
    verbose=1
)

Trial 10 Complete [00h 01m 56s]
val_accuracy: 0.5492178201675415

Best val_accuracy So Far: 0.7352586984634399
Total elapsed time: 00h 14m 30s


In [62]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:", best_hps.values)

Best Hyperparameters: {'num_layers': 5, 'units0': 64, 'activation0': 'sigmoid', 'dropout0': 0.1, 'l2_reg0': 0.0, 'kernel_init0': 'he_normal', 'optimizer': 'adam', 'units1': 128, 'activation1': 'sigmoid', 'dropout1': 0.4, 'l2_reg1': 0.0, 'kernel_init1': 'he_normal', 'units2': 72, 'activation2': 'relu', 'dropout2': 0.3, 'l2_reg2': 0.01, 'kernel_init2': 'he_uniform', 'units3': 128, 'activation3': 'relu', 'dropout3': 0.2, 'l2_reg3': 0.0, 'kernel_init3': 'he_uniform', 'units4': 8, 'activation4': 'sigmoid', 'dropout4': 0.4, 'l2_reg4': 0.1, 'kernel_init4': 'he_uniform'}


In [63]:
best_model = tuner.get_best_models(num_models=1)[0]

In [64]:
history = best_model.fit(
    X_train_ml.astype(np.float32),
    y_train.values.astype(np.float32),
    epochs=15,
    initial_epoch=5,  # continue from end of hyperparameter search
    validation_data=(X_test_ml.astype(np.float32), y_test.values.astype(np.float32))
)

Epoch 6/15
2078/2078 [==============================] - 22s 10ms/step - loss: 0.5388 - accuracy: 0.7198 - val_loss: 0.5275 - val_accuracy: 0.7346
Epoch 7/15
2078/2078 [==============================] - 22s 11ms/step - loss: 0.5371 - accuracy: 0.7222 - val_loss: 0.5225 - val_accuracy: 0.7330
Epoch 8/15
2078/2078 [==============================] - 20s 10ms/step - loss: 0.5360 - accuracy: 0.7239 - val_loss: 0.5208 - val_accuracy: 0.7309
Epoch 9/15
2078/2078 [==============================] - 20s 10ms/step - loss: 0.5357 - accuracy: 0.7235 - val_loss: 0.5314 - val_accuracy: 0.7090
Epoch 10/15
2078/2078 [==============================] - 20s 10ms/step - loss: 0.5355 - accuracy: 0.7230 - val_loss: 0.5190 - val_accuracy: 0.7362
Epoch 11/15
2078/2078 [==============================] - 21s 10ms/step - loss: 0.5335 - accuracy: 0.7256 - val_loss: 0.5176 - val_accuracy: 0.7374
Epoch 12/15
2078/2078 [==============================] - 21s 10ms/step - loss: 0.5335 - accuracy: 0.7232 - val_loss: 0.519

In [65]:
y_pred_proba = best_model.predict(X_test_ml.astype(np.float32)).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

520/520 [==============================] - 1s 2ms/step
Accuracy: 0.7379663056558363
F1 Score: 0.4546023794614903
Precision: 0.6156716417910447
Recall: 0.36033353186420486
ROC-AUC: 0.7818207853612525


In [66]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_ml, y_train = smote.fit_resample(X_train_ml, y_train.values)

In [67]:
X_train_ml.shape

(92658, 118)

In [69]:
import pandas as pd
print(pd.Series(y_train).value_counts())

0    46329
1    46329
Name: count, dtype: int64


In [70]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten

# Train-test split
X = df[["Strain Name","Antibiotic"]]
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Label encode Strain_Name for embedding
strain_le = LabelEncoder()
strain_le.fit(X_train["Strain Name"])
X_train_strain_idx = X_train["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values
X_test_strain_idx  = X_test["Strain Name"].map(lambda x: strain_le.transform([x])[0] if x in strain_le.classes_ else 0).values

# Simple embedding model
input_strain = Input(shape=(1,))
embed_layer = Embedding(input_dim=len(strain_le.classes_)+1, output_dim=100, name="strain_embedding")(input_strain)
embed_flat = Flatten()(embed_layer)
embedding_model = Model(inputs=input_strain, outputs=embed_flat)

# Forward pass to get embeddings
train_embeddings = embedding_model.predict(X_train_strain_idx)
test_embeddings  = embedding_model.predict(X_test_strain_idx)

print("Train embeddings shape:", train_embeddings.shape)
print("Test embeddings shape:", test_embeddings.shape)

520/520 [==============================] - 1s 1ms/step
Train embeddings shape: (66476, 100)
Test embeddings shape: (16620, 100)


In [72]:
from sklearn.preprocessing import OneHotEncoder

# Use sparse_output instead of sparse
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on training data
ohe.fit(X_train[["Antibiotic"]])

# Transform train and test
X_train_ohe = ohe.transform(X_train[["Antibiotic"]])
X_test_ohe  = ohe.transform(X_test[["Antibiotic"]])

# Optional: convert to DataFrame with column names
import pandas as pd
X_train_ohe = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_train.index)
X_test_ohe  = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(["Antibiotic"]), index=X_test.index)

In [73]:
# Concatenate embeddings with OHE features
X_train_ml = np.hstack([train_embeddings, X_train_ohe])
X_test_ml  = np.hstack([test_embeddings, X_test_ohe])

input_dim = X_train_ml.shape[1]  # This is the input_dim for MyHyperModel

In [74]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_ml, y_train = smote.fit_resample(X_train_ml, y_train.values)

In [82]:
input_dim = X_train_ml.shape[1]  

In [85]:
tuner = kt.RandomSearch(
    hypermodel,                    
    objective='val_accuracy',      
    max_trials=5,                 
    directory='mydir',
    project_name='project21'
)

In [83]:
hypermodel = MyHyperModel(input_dim=input_dim)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Input, Embedding, Flatten, Concatenate
from tensorflow.keras import regularizers, initializers
from kerastuner import HyperModel

# Example HyperModel class
class MyHyperModel(HyperModel):
    def __init__(self, input_dim):
        self.input_dim = input_dim

    def build(self, hp):
        model = Sequential()
        counter = 0
        for i in range(hp.Int('num_layers', min_value=1, max_value=5)):
            units = hp.Int('units' + str(i), min_value=8, max_value=128, step=8)
            activation = hp.Choice('activation' + str(i), values=['relu','tanh','sigmoid'])
            dropout_rate = hp.Choice('dropout' + str(i), values=[0.1,0.2,0.3,0.4,0.5])
            reg = regularizers.l2(hp.Choice('l2_reg'+str(i), values=[0.0, 0.001,0.01,0.1]))
            kernel_init = hp.Choice('kernel_init'+str(i), values=['glorot_uniform','he_uniform','he_normal','lecun_normal'])
            
            if counter == 0:
                model.add(
                    Dense(
                        units,
                        activation=activation,
                        input_dim=self.input_dim,
                        kernel_regularizer=reg,
                        kernel_initializer=kernel_init
                    )
                )
                model.add(BatchNormalization())
                model.add(Dropout(dropout_rate))
            else:
                model.add(
                    Dense(
                        units,
                        activation=activation,
                        kernel_regularizer=reg,
                        kernel_initializer=kernel_init
                    )
                )
                model.add(BatchNormalization())
                model.add(Dropout(dropout_rate))
            counter += 1

        model.add(Dense(1, activation='sigmoid'))
        optimizer = hp.Choice('optimizer', values=['rmsprop','adam','sgd','nadam','adadelta','adamw'])
        model.compile(optimizer=optimizer,
                      loss='binary_crossentropy',
                      metrics=['accuracy'])
        return model

In [86]:
tuner.search(
    X_train_ml.astype(np.float32),  # feature matrix
    y_train.astype(np.float32),  # target vector
    epochs=5,
    validation_data=(X_test_ml.astype(np.float32), y_test.astype(np.float32)),
    verbose=1
)

Trial 5 Complete [00h 00m 00s]

Best val_accuracy So Far: 0.6995186805725098
Total elapsed time: 00h 06m 50s


In [87]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("Best Hyperparameters:", best_hps.values)

Best Hyperparameters: {'num_layers': 5, 'units0': 56, 'activation0': 'sigmoid', 'dropout0': 0.2, 'l2_reg0': 0.01, 'kernel_init0': 'he_uniform', 'optimizer': 'rmsprop', 'units1': 72, 'activation1': 'sigmoid', 'dropout1': 0.5, 'l2_reg1': 0.01, 'kernel_init1': 'glorot_uniform', 'units2': 8, 'activation2': 'relu', 'dropout2': 0.1, 'l2_reg2': 0.0, 'kernel_init2': 'glorot_uniform', 'units3': 8, 'activation3': 'relu', 'dropout3': 0.1, 'l2_reg3': 0.0, 'kernel_init3': 'glorot_uniform', 'units4': 8, 'activation4': 'relu', 'dropout4': 0.1, 'l2_reg4': 0.0, 'kernel_init4': 'glorot_uniform'}


In [88]:
best_model = tuner.get_best_models(num_models=1)[0]

In [90]:
history = best_model.fit(
    X_train_ml.astype(np.float32),
    y_train.astype(np.float32),
    epochs=15,
    initial_epoch=5,  # continue from end of hyperparameter search
    validation_data=(X_test_ml.astype(np.float32), y_test.values.astype(np.float32))
)

Epoch 6/15
2896/2896 [==============================] - 41s 14ms/step - loss: 0.6116 - accuracy: 0.6957 - val_loss: 0.5995 - val_accuracy: 0.6854
Epoch 7/15
2896/2896 [==============================] - 37s 13ms/step - loss: 0.6106 - accuracy: 0.6968 - val_loss: 0.5825 - val_accuracy: 0.6975
Epoch 8/15
2896/2896 [==============================] - 37s 13ms/step - loss: 0.6087 - accuracy: 0.6983 - val_loss: 0.6018 - val_accuracy: 0.6835
Epoch 9/15
2896/2896 [==============================] - 42s 14ms/step - loss: 0.6098 - accuracy: 0.6984 - val_loss: 0.6120 - val_accuracy: 0.6735
Epoch 10/15
2896/2896 [==============================] - 37s 13ms/step - loss: 0.6050 - accuracy: 0.6974 - val_loss: 0.6247 - val_accuracy: 0.6496
Epoch 11/15
2896/2896 [==============================] - 37s 13ms/step - loss: 0.6051 - accuracy: 0.6984 - val_loss: 0.5899 - val_accuracy: 0.6866
Epoch 12/15
2896/2896 [==============================] - 39s 14ms/step - loss: 0.6055 - accuracy: 0.6990 - val_loss: 0.593

In [91]:
y_pred_proba = best_model.predict(X_test_ml.astype(np.float32)).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

520/520 [==============================] - 1s 2ms/step
Accuracy: 0.6885078219013238
F1 Score: 0.5892247877489487
Precision: 0.49074808353158866
Recall: 0.7371451260671035
ROC-AUC: 0.7688092403531488


In [93]:
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

# --- Train-test split
X_train, X_test, y_train, y_test = train_test_split(df[["Strain Name","Antibiotic"]],
                                                    df["target"],
                                                    test_size=0.2,
                                                    stratify=df["target"],
                                                    random_state=42)

# ===========================
# Step 1: Train Word2Vec on Strain Names
# ===========================
# Word2Vec expects tokenized sentences. We'll treat each "Strain_Name" as a list of characters/words
sentences = [[str(x)] for x in X_train["Strain Name"]]
w2v_model = Word2Vec(sentences, vector_size=50, window=1, min_count=1, workers=2, seed=42)

# Get embeddings for each sample by averaging word embeddings (here just single token)
def get_w2v_embedding(token):
    if token in w2v_model.wv:
        return w2v_model.wv[token]
    else:
        return np.zeros(w2v_model.vector_size)

X_train_w2v = np.array([get_w2v_embedding(t) for t in X_train["Strain Name"]])
X_test_w2v  = np.array([get_w2v_embedding(t) for t in X_test["Strain Name"]])

# --- One-hot encode Antibiotic
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Fit on training data
ohe.fit(X_train[["Antibiotic"]])

# Transform train and test
X_train_ohe = ohe.transform(X_train[["Antibiotic"]])
X_test_ohe  = ohe.transform(X_test[["Antibiotic"]])

# Convert to DataFrame with proper column names
X_train_ohe = pd.DataFrame(
    X_train_ohe,
    columns=ohe.get_feature_names_out(["Antibiotic"]),
    index=X_train.index
)
X_test_ohe = pd.DataFrame(
    X_test_ohe,
    columns=ohe.get_feature_names_out(["Antibiotic"]),
    index=X_test.index
)

# --- Concatenate W2V + OHE
X_train_final = np.hstack([X_train_w2v, X_train_ohe])
X_test_final  = np.hstack([X_test_w2v, X_test_ohe])

# ===========================
# Step 2: Simple MLP
# ===========================
input_dim = X_train_final.shape[1]
model = Sequential([
    Dense(32, activation='relu', input_dim=input_dim),
    BatchNormalization(),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# ===========================
# Step 3: Train
# ===========================
history = model.fit(X_train_final.astype(np.float32),
                    y_train.values.astype(np.float32),
                    validation_data=(X_test_final.astype(np.float32), y_test.values.astype(np.float32)),
                    epochs=20,
                    batch_size=4,
                    verbose=1)

# ===========================
# Step 4: Evaluate
# ===========================
y_pred_proba = model.predict(X_test_final.astype(np.float32)).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

Epoch 1/20
16619/16619 [==============================] - 67s 4ms/step - loss: 0.5603 - accuracy: 0.7092 - val_loss: 0.5333 - val_accuracy: 0.7277
Epoch 2/20
16619/16619 [==============================] - 77s 5ms/step - loss: 0.5395 - accuracy: 0.7174 - val_loss: 0.5277 - val_accuracy: 0.7289
Epoch 3/20
16619/16619 [==============================] - 68s 4ms/step - loss: 0.5333 - accuracy: 0.7181 - val_loss: 0.5369 - val_accuracy: 0.7304
Epoch 4/20
16619/16619 [==============================] - 75s 5ms/step - loss: 0.5302 - accuracy: 0.7215 - val_loss: 0.5297 - val_accuracy: 0.7313
Epoch 5/20
16619/16619 [==============================] - 67s 4ms/step - loss: 0.5298 - accuracy: 0.7210 - val_loss: 0.5271 - val_accuracy: 0.7334
Epoch 6/20
16619/16619 [==============================] - 69s 4ms/step - loss: 0.5288 - accuracy: 0.7227 - val_loss: 0.5306 - val_accuracy: 0.7344
Epoch 7/20
16619/16619 [==============================] - 65s 4ms/step - loss: 0.5281 - accuracy: 0.7224 - val_loss: 0